# Simple U-Net Denoiser (STL-10, native 96x96, single objective)

Same simple, single-`"x0"`-objective denoising setup as
`unet_denoising_simple.ipynb` (CIFAR-10 @ 64x64), moved to STL-10's
native **96x96x3** resolution (no resizing/cropping), using the
`SmallUNet` architecture already defined in `code/stl10_unet.py` (3
encoder/decoder levels, 96->48->24->12, convolutional bottleneck, no FC
layer) instead of `TinyUNet64` (which was sized for 64x64).

`SmallUNet` was originally built for **plain** reconstruction (no
corruption): `L = ||x - SmallUNet(x)||^2`. Reused here for
**denoising** by corrupting the input first:

```
x_noisy = x + sigma*eps,   eps ~ N(0,I),   sigma = noise_std
L = ||x - SmallUNet(x_noisy)||^2
```

No change to the model itself -- `SmallUNet` already ends in a Sigmoid
and outputs `x_hat` directly, so only the train/eval loop (in
`code/unet_denoising_simple_stl10.py`) differs from `stl10_unet.py`'s
own: it corrupts the input before the forward pass, while the loss
target stays the clean image.

As in the CIFAR-10 version, because there is only one objective here,
`train_loss` and `test_denoising_mse` are the same formula
(`MSE(x_hat, x)`) computed on different splits (train, with gradients
on and parameters changing mid-epoch, vs. held-out test, frozen model)
-- see `unet_denoising_simple.ipynb` for the fuller explanation.

Core code lives in `code/unet_denoising_simple_stl10.py`
(version-controlled); this notebook is a thin orchestration layer: GPU
check, repo pull, build model + print the full shape trace, download
real STL-10 (cached in Drive), train, plot the loss curve, and show
clean/noisy/denoised qualitative + quantitative results as the required
results-visualization step.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model code lives in `code/stl10_unet.py` (architecture, reused as-is)
and `code/unet_denoising_simple_stl10.py` (denoising train/eval loop,
visualization), not duplicated in this notebook. Replace `REPO_URL`
with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model, print full shape trace

One verbose forward pass on a small synthetic 96x96 batch, printing the
`[B,C,H,W]` shape after every op -- including, per level, each skip
tensor's shape at the moment it is saved and the shapes immediately
before/after each `torch.cat`. Not part of training.

In [ ]:
from stl10_unet import SmallUNet, count_trainable_parameters

torch.manual_seed(0)

model = SmallUNet().to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 3, 96, 96, device=device)  # [B,3,96,96], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)


## 4. Download real STL-10 (train/test split), native 96x96, cached in Drive

`get_stl10_dataloaders` (from `code/stl10_unet.py`) uses `torchvision`'s
native labeled train/test partition at STL-10's native 96x96
resolution -- no resizing. Labels are unused (the denoising target is
the clean image itself). STL-10 is a ~2.6GB download, so Drive caching
matters even more here than for CIFAR-10.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so we mount Google Drive and point `DATA_ROOT` at a
persistent Drive folder: `torchvision` downloads into it once and
reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/stl10"


## 5. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`, both defined
in `code/unet_denoising_simple_stl10.py`): each clean batch is
corrupted with additive Gaussian noise (`noise_std`) and `SmallUNet` is
trained to recover the clean image, MSE loss, Adam optimizer.

In [ ]:
import torch.nn as nn
from unet_denoising_simple_stl10 import train_one_epoch, evaluate
from stl10_unet import get_stl10_dataloaders

BATCH_SIZE = 64
N_EPOCHS = 15
LR = 1e-3
NOISE_STD = 0.3  # std of additive Gaussian noise defining the denoising task

train_loader, test_loader = get_stl10_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "test_denoising_mse": []}

for epoch in range(N_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device, NOISE_STD)
    test_mse = evaluate(model, test_loader, loss_fn, device, NOISE_STD)

    history["train_loss"].append(train_loss)
    history["test_denoising_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_loss={train_loss:.4f}  test_denoising_mse={test_mse:.4f}")


## 6. Loss curve (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="train_loss")
ax.plot(history["test_denoising_mse"], label="test_denoising_mse")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE(x_hat, clean)")
ax.set_title("Denoising loss")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_denoising_mse = {history['test_denoising_mse'][-1]:.4f}")


## 7. Qualitative + quantitative check: clean vs. noisy vs. denoised

Per-project convention: always look at concrete examples, not only
scalar metrics. Shows 5 real test images as clean (row 1) / noisy input
(row 2) / SmallUNet output (row 3), with per-image MSE and PSNR (output
vs. clean) printed under each denoised image.

In [ ]:
from unet_denoising_simple_stl10 import visualize_denoising

sample_images, _sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_denoising(model, sample_images, NOISE_STD, num_examples=5)


## 8. Final test-set denoising MSE

In [ ]:
final_test_mse = evaluate(model, test_loader, loss_fn, device, NOISE_STD)
print(f"average test-set denoising MSE (noise_std={NOISE_STD}): {final_test_mse:.4f}")
